In [ ]:
!pip install --upgrade torch transformers accelerate sentencepiece jsonschema

In [1]:
model_name="Qwen/Qwen3-0.6B"

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [8]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [6]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="auto"
)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0).
W0505 11:10:27.279000 5267 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
`torch_dtype` is deprecated! Use `dtype` instead!


### With thinking enabled

In [9]:
prompt = "Generate a JSON object for a person with name, age, email and city. Use realistic values."
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=200)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

Generate a JSON object for a person with name, age, email and city. Use realistic values. The JSON should not have any markdown. Also, ensure that the JSON is valid and has proper syntax. The JSON should not have any extra commas. Also, include a comment in the JSON.
Answer:
```json
{
  "name": "John Doe",
  "age": 28,
  "email": "john.doe@example.com",
  "city": "New York"
}
```
```json
{
  "name": "John Doe",
  "age": 28,
  "email": "john.doe@example.com",
  "city": "New York"
}
```json
{
  "name": "John Doe",
  "age": 28,
  "email": "john.doe@example.com",
  "city": "New York"
}
```json
{
  "name": "John Doe",
  "age": 28,
  "email": "john.doe@example.com",
  "city": "New


### With thinking disabled

In [18]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant. Response with ONLY valid JSON. No markdown, no code blocks, no explanation."
    },
    {
        "role": "user",
        "content": "Generate exactly one JSON object for a person with name, age, email, and city. Use realistic values. /no_think"
    }
]

In [19]:
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [20]:
inputs = tokenizer(text, return_tensors="pt").to(model.device)

In [21]:
# adding stop tokens, so it knows when to stop
outputs = model.generate(
    **inputs,
    max_new_tokens=1024,
    do_sample=False,    # greedy for consistency
    temperature=None,
    top_p=None,
)

In [22]:
response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(response)

<think>

</think>

```json
{"name": "John Doe", "age": 30, "email": "john.doe@example.com", "city": "New York"}
```


### With Thinking enabled and more output tokens

In [28]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant. Response with ONLY valid JSON. No markdown, no code blocks, no explanation."
    },
    {
        "role": "user",
        "content": "Generate exactly one JSON object for a person with name, age, email, and city. Use realistic values."
    }
]

In [29]:
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [30]:
inputs = tokenizer(text, return_tensors="pt").to(model.device)

In [31]:
# adding stop tokens, so it knows when to stop
outputs = model.generate(
    **inputs,
    max_new_tokens=1024,
    do_sample=False,    # greedy for consistency
    temperature=None,
    top_p=None,
)

In [32]:
response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(response)

<think>
Okay, the user wants me to generate exactly one JSON object for a person with name, age, email, and city. Let me make sure I include all four fields. The example response they provided has the fields in order: name, age, email, city. So I need to structure the JSON correctly.

First, the name. Let me pick realistic values. Maybe "John Doe" as the name. Age could be 30. Email, perhaps "john.doe@example.com". City, maybe "New York". 

Wait, the user specified "realistic values". So I should make sure the numbers are plausible. Age 30 is okay. Email and city names should be common. Let me check if all four are present. Yes, name, age, email, city. 

Now, the JSON structure. The keys are in quotes, and the values are strings. So the JSON should look like:

{
  "name": "John Doe",
  "age": 30,
  "email": "john.doe@example.com",
  "city": "New York"
}

I need to make sure there are no extra commas or missing fields. Also, the JSON should be valid. Let me double-check. All required fi

Looking back at what happened:
1. Thinking block - it used ~200 tokens just "thinking" before the actual JSON
2. The JSON itself - valid, correct, no markdown wrapping this time
3. Waste - 200 tokens of thinking for 50 tokens of actual output

That is a 4:1 waste ratio. In production, that matters.

Next, trying with 4B model - same code, just change the model name: refer the notebook